# Python Break & Continue

> **Python Mastery** · Module 02 — Control Flow · Lesson 5/5

Three small keywords - `break`, `continue`, `pass` - decide what a loop does *in the middle* of a pass: quit it, skip one round, or do nothing at all. They look interchangeable to beginners; by the end of this lesson you'll know exactly where each one sends execution and when each makes code *more* readable.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Trace** exactly where execution jumps for `break`, `continue`, and `pass`
- **Choose** between them using a side-by-side comparison
- **Predict** that `break` inside nested loops exits only the innermost loop
- **Implement** the search pattern: find the first match with a flag or `break`
- **Use** loop-`else` with `break` to express found / not-found cleanly
- **Judge** when each keyword improves readability rather than hurting it

## 1. `break`: Leave the Loop Now

The moment Python hits `break`, the loop is abandoned - no further items, no condition re-check - and execution resumes on the first line **after** the loop body.

Typical jobs: stop at the first match, abort on invalid data, exit a retry loop after success.

**Syntax:**
```python
for item in items:
    ...
    if exit_condition:
        break          # jump PAST the loop
    ...rest of body...
```

In [ ]:
seats = ["taken", "empty", "taken", "empty"]

for i, seat in enumerate(seats, start=1):
    if seat == "empty":
        print(f"Sitting in seat {i}")
        break                       # stop scanning, we have a seat
    print(f"Seat {i}: occupied, moving on...") 

print("Loop is over.")                 # break lands HERE

Seat 1: occupied, moving on...
Sitting in seat 2
Loop is over.


## 2. `continue`: Skip This Item

`continue` cancels only the **current pass**: everything below it in the body is skipped, and control jumps back to the top of the loop (re-test the condition for `while`, fetch the next item for `for`). The loop itself keeps going.

Typical job: guard clauses - *"if this record isn't worth processing, move along."*

**Syntax:**
```python
for item in items:
    if not worth_processing(item):
        continue         # jump to the NEXT iteration
    ...real work...
```

In [2]:
orders = [
    {"id": 101, "amount": 500},
    {"id": 102, "amount": 0},     # cancelled -> amount 0
    {"id": 103, "amount": 1250},
]

total = 0
for order in orders:
    if order["amount"] <= 0:
        continue                    # skip cancelled orders entirely
    total += order["amount"]
    print(f"Charged #{order['id']}: {order['amount']} BDT")

print("Revenue:", total, "BDT")

Charged #101: 500 BDT
Charged #103: 1250 BDT
Revenue: 1750 BDT


## 3. `pass`: Do Nothing, Legally

`pass` is not about loops at all - it's a placeholder that satisfies Python's *"a block must contain something"* rule while doing absolutely nothing. Execution simply falls through to the next statement **inside the same block** (it does NOT skip the rest of the loop pass like `continue`).

Common homes: stub functions/classes you'll fill in later, and branches you've deliberately decided need no action.

**Syntax:**
```python
if error_is_expected:
    pass             # consciously ignore this case
else:
    handle(error)
```

In [3]:
error_code = 404

if error_code == 404:
    pass                          # missing pages are expected; ignore them
else:
    print("Unexpected error:", error_code)

print("Server keeps running.")

class FutureFeature:              # stub class - very common use
    pass

Server keeps running.


## 4. Side-by-Side: Who Jumps Where?

| Keyword | Execution jumps to | Loop continues? | Rest of this pass runs? | Typical use |
|---|---|---|---|---|
| `break` | first line AFTER the loop | No | No | stop at first match / abort |
| `continue` | next iteration test (top of loop) | Yes | No | filter out unwanted items |
| `pass` | the very next statement | Yes | **Yes** | placeholder; deliberate no-op |

Same loop three ways - watch the outputs differ:

In [4]:
numbers = [1, 3, 4, 5, 7]

print("-- break at first even --")
for n in numbers:
    if n % 2 == 0:
        print("even found:", n)
        break
    print("odd:", n)

print("-- continue skips evens --")
for n in numbers:
    if n % 2 == 0:
        continue
    print("odd:", n)

print("-- pass changes nothing --")
for n in numbers:
    if n % 2 == 0:
        pass                      # placeholder; still printed below
    print("saw:", n)

-- break at first even --
odd: 1
odd: 3
even found: 4
-- continue skips evens --
odd: 1
odd: 3
odd: 5
odd: 7
-- pass changes nothing --
saw: 1
saw: 3
saw: 4
saw: 5
saw: 7


> 🔍 **Under the Hood:** `break` and `continue` bind to the **innermost enclosing loop** - there's no label syntax like Java's `break outer;`. CPython compiles each keyword into an explicit jump instruction targeting the block it sits in, so escaping two levels requires either a flag checked by both loops, or wrapping the inner loop in a function so `return` does the escaping. That function trick is the cleanest escape hatch Python offers.

In [ ]:
grid = [
    [2, 4, 6],
    [8, 9, 10],
]

for row_i, row in enumerate(grid):
    print("Row", row_i)
    for value in row:
        if value % 2 != 0:
            print("  odd value", value, "-> abandoning THIS row only")
            break                # exits the INNER loop; outer keeps going
        print("  even:", value)

print()
print("Escaping BOTH loops with a function + return:")

def first_odd_position(grid):
    for row_i, row in enumerate(grid):
        for col_i, value in enumerate(row):
            if value % 2 != 0:
                return row_i, col_i      # returns from BOTH loops
    return None

print(first_odd_position([[2, 4], [6, 9]]))
de

Row 0
  even: 2
  even: 4
  even: 6
Row 1
  even: 8
  odd value 9 -> abandoning THIS row only

Escaping BOTH loops with a function + return:
(1, 1)


## 6. The Search Pattern: Find the First Match

Half of all `break`s belong to the same job: walk a collection, stop at the first item satisfying a test. Two idiomatic shapes:

1. **Result variable + break** - keep the winner, stop looking.
2. **Loop-else** - let the loop finish naturally; `else` means *"nothing matched."*

Both beat the beginner version that scans the whole list even after finding the answer.

In [6]:
products = [
    ("keyboard", 1500),
    ("mouse", 700),
    ("monitor", 9800),
    ("webcam", 2400),
]
budget = 2000

# Idiom 1: result variable + break
found = None
for name, price in products:
    if price <= budget:
        found = (name, price)
        break                  # first affordable product wins

if found:
    print(f"Cheapest-in-budget pick: {found[0]} at {found[1]} BDT")

# Idiom 2: for...else - no flag needed
target = "monitor"
for name, price in products:
    if name == target:
        print(target, "is in stock at", price, "BDT")
        break
else:
    print(target, "is out of stock")

Cheapest-in-budget pick: keyboard at 1500 BDT
monitor is in stock at 9800 BDT


## 7. When Each Keyword Improves Readability

Used well, these keywords remove nesting and make intent obvious:

- **`continue` as a guard clause** - reject bad rows at the top of the body so the main logic stays un-indented and flat.
- **`break` to say *stop looking*** - far clearer than a `while running:` flag nobody can trace.
- **`pass` to say *nothing belongs here yet*** - documents a deliberate gap.

Used badly they hide control flow:

- Multiple scattered `break`s inside long bodies force readers to hunt for every exit.
- A `continue` in a `while` loop that sits above the counter update = infinite loop (see Lesson 03).
- `pass` where real logic was meant silently does nothing - the bug ships without an error.

Rule of thumb: **one clear exit per loop** reads best; if you need four, split the loop into a function.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Expecting `break` to exit all nested loops | It leaves only the innermost loop | Use a function + `return`, or a shared flag |
| Writing `continue` above the counter update in a `while` | Update skipped forever → infinite loop | Move the update before any `continue` |
| Confusing `pass` with `continue` or with deleting a branch | Code below `pass` still runs; branch stays | Use `continue` to skip, remove the block to delete |
| Using `break` outside any loop | `SyntaxError: 'break' outside loop` | Only legal inside `for`/`while` bodies |
| Assuming loop-`else` always runs | It's skipped whenever a `break` fired | Remember: else means "no break happened" |

## 💡 Best Practices & Pro Tips

- **Prefer guards over deep negated ifs**: `if bad: continue` beats `if not bad: <everything indented>`.
- **Name what you broke out of**: extracting the loop into a function and returning a result (`find_first_affordable(...)`) usually communicates better than `break` + leftover variables.
- **Keep one exit point when possible**, but don't be dogmatic - a single early `break` is clearer than a tangled boolean flag.
- **Comment the *why* of a `pass`**: `pass` alone looks like unfinished work; `pass  # 404s are expected here` looks like a decision.
- **AI-engineering relevance:** these keywords are how pipelines stay cheap - `continue` past malformed samples during preprocessing, `break` out of an evaluation loop once accuracy stops improving, wrap retry logic around flaky model-API calls. Every wasted GPU-minute or paid API call you skip is money saved.

## 📌 Summary

| Keyword | What it does | Example |
|---|---|---|
| `break` | exits the innermost loop immediately | `if hit: break` |
| `continue` | jumps to the next iteration | `if invalid: continue` |
| `pass` | syntactic no-op placeholder | `class Stub: pass` |
| loop `else` | runs only when no `break` fired | search found/not-found idiom |

Key takeaways:

- `break` → leave the whole loop; `continue` → leave just this pass; `pass` → change nothing.
- Nested loops: `break`/`continue` touch **only the innermost** loop - use a function to escape deeper.
- First-match searches read best as result-variable + `break`, or loop-`else` for the not-found case.
- Every keyword should earn its place by removing nesting, not adding mystery.

## 🔗 Next Lesson

Module 02 complete! Next module: **[03_Functions](../../03_Functions/)** - packaging your logic into reusable, nameable blocks with `def`.